# RF Model Training

In [26]:
# 导入必要的库
from tqdm import tqdm
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import os

# ----------------------------
# 1. 加载目标变量 y
# ----------------------------
buchwald_path = '/data/coding/Buchwald/Buchwald.csv'
df_buchwald = pd.read_csv(buchwald_path)
y = df_buchwald['Output'].values / 100  # 归一化输出

# ----------------------------
# 2. 定义超参数
# ----------------------------
random_states = [1, 2, 3, 4, 42]
n_estimators_list = [100, 200, 300, 500, 1000]
max_depths = [10, 50, 100, 200, 500]

# ----------------------------
# 3. 要处理的文件列表
# ----------------------------
files = ['Additive_new.csv', 'Aryl_halide_new.csv', 'Base_new.csv', 'Ligand_new.csv', 'Product_new.csv', 'Additive.csv', 'Aryl_halide.csv', 'Base.csv', 'Ligand.csv', 'Product.csv']
base_path = '/data/coding/Buchwald'

# ----------------------------
# 4. 遍历每个文件
# ----------------------------
for file in files:
    file_path = os.path.join(base_path, file)
    print(f"正在处理文件: {file}")
    
    # 读取特征数据
    df = pd.read_csv(file_path)
    X = df.values
    
    # 创建结果存储结构：每个 (n_estimators, max_depth) 组合保存多个 random_state 的结果
    summary_data = []

    # 遍历每一对 (n_estimators, max_depth)
    for n_est in n_estimators_list:
        print(f"树木为{n_est} --正在训练--")
        for max_d in tqdm(max_depths):
            r2_scores = []
            rmse_scores = []
            mae_scores = []

            # 遍历每个 random_state，训练并评估
            for rs in random_states:
                rf = RandomForestRegressor(n_estimators=n_est, max_depth=max_d, random_state=rs, n_jobs=-1)
                rf.fit(X, y)
                y_pred = rf.predict(X)

                # 计算评估指标
                r2 = r2_score(y, y_pred)
                mse = mean_squared_error(y, y_pred)
                rmse = np.sqrt(mse)
                mae = mean_absolute_error(y, y_pred)

                r2_scores.append(r2)
                rmse_scores.append(rmse)
                mae_scores.append(mae)

            # 计算均值和标准差
            r2_mean, r2_std = np.mean(r2_scores), np.std(r2_scores)
            rmse_mean, rmse_std = np.mean(rmse_scores), np.std(rmse_scores)
            mae_mean, mae_std = np.mean(mae_scores), np.std(mae_scores)

            # 格式化为 "mean ± std" 字符串，保留4位小数
            r2_str = f"{r2_mean:.4f} ± {r2_std:.4f}"
            rmse_str = f"{rmse_mean:.4f} ± {rmse_std:.4f}"
            mae_str = f"{mae_mean:.4f} ± {mae_std:.4f}"

            # 保存这一组超参数的结果
            summary_data.append({
                'n_estimators': n_est,
                'max_depth': max_d,
                'R2': r2_str,
                'RMSE': rmse_str,
                'MAE': mae_str
            })

    # 转为 DataFrame
    results_df = pd.DataFrame(summary_data)

    # 保存
    results_df.to_csv(f'/data/coding/RF_data/RF_{file}', index=False)

    break

正在处理文件: Additive_new.csv
树木为100 --正在训练--


  0%|          | 0/5 [00:00<?, ?it/s]

100%|██████████| 5/5 [00:04<00:00,  1.19it/s]


树木为200 --正在训练--


100%|██████████| 5/5 [00:07<00:00,  1.46s/it]


...

100%|██████████| 5/5 [00:10<00:00,  2.07s/it]


树木为500 --正在训练--


100%|██████████| 5/5 [00:16<00:00,  3.33s/it]


树木为1000 --正在训练--


100%|██████████| 5/5 [00:31<00:00,  6.38s/it]


# RF Model Training with Selected Mordred Features (500)

In [29]:
# 导入必要的库
from tqdm import tqdm
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import os

# ----------------------------
# 1. 加载目标变量 y
# ----------------------------
buchwald_path = '/data/coding/Buchwald/Buchwald.csv'
df_buchwald = pd.read_csv(buchwald_path)
y = df_buchwald['Output'].values / 100  # 归一化输出

# ----------------------------
# 2. 定义超参数
# ----------------------------
random_states = [1, 2, 3, 4, 42]
n_estimators_list = [100, 200, 300, 500, 1000]
max_depths = [10, 50, 100, 200, 500]

# ----------------------------
# 3. 要处理的文件列表
# ----------------------------
files = ['Additive_new.csv', 'Aryl_halide_new.csv', 'Base_new.csv', 'Ligand_new.csv', 'Product_new.csv']
base_path = '/data/coding/Buchwald'

# ----------------------------
# 4. 遍历每个文件
# ----------------------------
dfs = []
for file in files:
    file_path = os.path.join(base_path, file)
    df = pd.read_csv(file_path, index_col=0)
    dfs.append(df)

# 读取特征数据
combined_df = pd.concat(dfs, axis=1, ignore_index=False)
X = combined_df.values

# 划分数据集
X_train, X_test, y_train, y_test = train_test_split(X, y,test_size=0.3,random_state =42)

# 创建结果存储结构：每个 (n_estimators, max_depth) 组合保存多个 random_state 的结果
summary_data = []

 # 遍历每个 random_state，训练并评估
for rs in random_states:
    #划分数据集
    X_train, X_test, y_train, y_test = train_test_split(X, y,test_size=0.3,random_state =rs)

    # 遍历每一对 (n_estimators, max_depth)
    for n_est in n_estimators_list:
        print(f"树木为{n_est} --正在训练--")
        for max_d in tqdm(max_depths):
            r2_scores = []
            rmse_scores = []
            mae_scores = []

            rf = RandomForestRegressor(n_estimators=n_est, max_depth=max_d, random_state=rs, n_jobs=-1)
            rf.fit(X_train, y_train)
            y_pred = rf.predict(X_test)

            # 计算评估指标
            r2 = r2_score(y_test, y_pred)
            mse = mean_squared_error(y_test, y_pred)
            rmse = np.sqrt(mse)
            mae = mean_absolute_error(y_test, y_pred)

            # 保存这一组超参数的结果
            summary_data.append({
                'random_state': rs,
                'n_estimators': n_est,
                'max_depth': max_d,
                'R2': r2,
                'RMSE': rmse,
                'MAE': mae
            })

    # 转为 DataFrame
    results_df = pd.DataFrame(summary_data)

    # 保存
    results_df.to_csv(f'/data/coding/RF_data/FullRF.csv', index=False)

树木为100 --正在训练--


100%|██████████| 5/5 [00:04<00:00,  1.03it/s]


树木为200 --正在训练--


100%|██████████| 5/5 [00:08<00:00,  1.70s/it]


树木为300 --正在训练--


...

100%|██████████| 5/5 [00:12<00:00,  2.54s/it]


树木为500 --正在训练--


100%|██████████| 5/5 [00:20<00:00,  4.19s/it]


树木为1000 --正在训练--


100%|██████████| 5/5 [00:40<00:00,  8.10s/it]
